# 🎯 Pokémon Kaggle Competition: Baseline Model (Triple-Modal PyTorch)
Welcome to the Pokémon Kaggle Competition! 
In this notebook, we will build a **Triple-Modal Neural Network** using PyTorch. 
Our goal is to predict if a Pokémon is **Legendary** (1 = Yes, 0 = No) by combining three types of data:
1. **Numerical Stats:** (HP, Attack, Defense, etc.)
2. **Image Data:** (The official sprite of the Pokémon)
3. **Text Data:** (The Pokémon's Name)

Let's build a Neural Network that handles all three!

In [ ]:
import pandas as pd
import numpy as np
import os
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.preprocessing import StandardScaler

## 1. Prepare Dataset Class & Text Tokenizer
In PyTorch, we need a custom `Dataset` class. We will also create a simple character-level tokenizer to convert Pokémon names into padded sequences of numbers.

In [ ]:
# 1. Load Data
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')

# 2. Prepare Text Tokenizer (Character-level)
# Extract all unique characters from training names
all_chars = set(''.join(train_df['Name'].str.lower()))
char2idx = {c: i+1 for i, c in enumerate(all_chars)}
char2idx['<PAD>'] = 0 # Reserve 0 for padding
vocab_size = len(char2idx)
max_name_len = 15

def encode_name(name):
    # Convert string to list of integers and pad to max_len
    encoded = [char2idx.get(c, 0) for c in name.lower()]
    if len(encoded) < max_name_len:
        encoded += [0] * (max_name_len - len(encoded))
    return encoded[:max_name_len]

# 3. Custom PyTorch Dataset
num_features = ['HP', 'Attack', 'Defense', 'Sp_Atk', 'Sp_Def', 'Speed', 'Weight', 'Height']

class PokemonDataset(Dataset):
    def __init__(self, df, img_dir, scaler=None, is_train=True):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.is_train = is_train
        
        # Scale numerical features
        self.scaler = scaler
        if self.scaler is None:
            self.scaler = StandardScaler()
            self.scaled_num = self.scaler.fit_transform(self.df[num_features])
        else:
            self.scaled_num = self.scaler.transform(self.df[num_features])
            
        self.transform = transforms.Compose([
            transforms.Resize((64, 64)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # A. Load Image
        img_name = os.path.join(self.img_dir, self.df.iloc[idx]['Image_File'])
        image = Image.open(img_name).convert('RGB')
        image = self.transform(image)
        
        # B. Load Stats
        stats = torch.tensor(self.scaled_num[idx], dtype=torch.float32)
        
        # C. Load Text (Name)
        text_seq = torch.tensor(encode_name(self.df.iloc[idx]['Name']), dtype=torch.long)
        
        if self.is_train:
            label = torch.tensor(self.df.iloc[idx]['Is_Legendary'], dtype=torch.float32)
            return image, stats, text_seq, label
        else:
            return image, stats, text_seq

# Create Datasets and DataLoaders
train_dataset = PokemonDataset(train_df, 'data/images/')
test_dataset = PokemonDataset(test_df, 'data/images/', scaler=train_dataset.scaler, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
print(f"Train size: {len(train_dataset)} | Test size: {len(test_dataset)}")
print(f"Vocab size: {vocab_size}")

## 2. Build the Triple-Modal Neural Network
We will create a custom `nn.Module` with 3 branches:
- **Image Branch:** Flattens the 64x64 image.
- **Stats Branch:** Processes numerical stats.
- **Text Branch:** Uses `nn.Embedding` to process the character sequence of the name.
- **Merge:** Concatenates all 3 outputs into a final prediction.

In [ ]:
class TripleModalNN(nn.Module):
    def __init__(self, num_stats_features, vocab_size, max_name_len):
        super(TripleModalNN, self).__init__()
        
        # 1. Image branch (64x64x3 = 12288)
        self.img_branch = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 64 * 3, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        )
        
        # 2. Stats branch
        self.stats_branch = nn.Sequential(
            nn.Linear(num_stats_features, 32),
            nn.ReLU()
        )
        
        # 3. Text branch (Embedding)
        self.text_branch = nn.Sequential(
            nn.Embedding(num_embeddings=vocab_size, embedding_dim=16),
            nn.Flatten(),
            nn.Linear(max_name_len * 16, 32),
            nn.ReLU()
        )
        
        # Merged classification head (64 + 32 + 32 = 128)
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid() # Output probability between 0 and 1
        )
        
    def forward(self, image, stats, text):
        img_features = self.img_branch(image)
        stats_features = self.stats_branch(stats)
        text_features = self.text_branch(text)
        
        # Concatenate all features
        merged = torch.cat((img_features, stats_features, text_features), dim=1)
        output = self.classifier(merged)
        return output

model = TripleModalNN(num_stats_features=len(num_features), vocab_size=vocab_size, max_name_len=max_name_len)
criterion = nn.BCELoss() # Binary Cross Entropy
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(model)

In [ ]:
# Train the model
epochs = 15
model.train()

for epoch in range(epochs):
    epoch_loss = 0
    correct = 0
    total = 0
    
    for images, stats, texts, labels in train_loader:
        optimizer.zero_grad()
        
        outputs = model(images, stats, texts).squeeze()
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        # Calculate accuracy
        preds = (outputs > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss/len(train_loader):.4f} | Acc: {correct/total:.4f}")

## 3. Generate Submission File

In [ ]:
# Predict on test set
model.eval()
test_preds = []

with torch.no_grad():
    for images, stats, texts in test_loader:
        outputs = model(images, stats, texts).squeeze()
        
        # Handle edge case where batch size is 1 (outputs becomes scalar)
        if outputs.dim() == 0:
            outputs = outputs.unsqueeze(0)
            
        preds = (outputs > 0.5).int().numpy()
        test_preds.extend(preds)

# Prepare submission dataframe
submission = pd.read_csv('data/sample_submission.csv')
submission['Is_Legendary'] = test_preds

# Save to CSV
submission.to_csv('submission.csv', index=False)
print("submission.csv successfully created! Ready to upload to Kaggle.")